# v11 — Day-Invariant Overfit

**Insight from v10:**  
- `(geohash, hour, minute)` cross-day mean_std = 0.00408 → demand is nearly day-invariant  
- v10 Day48→Day49 R² dropped to 0.75 only because I included `day` as a feature, letting the model make day-specific splits that don't generalize to day 49 in test  

**v11 strategy:**  
1. Build features **without** `day` (so model can't split on it)  
2. Sanity-check: train overfit on day 48 only, hold out day 49 — R² should now be HIGH  
3. If sanity check passes, refit on FULL train, predict test, submit  

**Decision rule for Cell In[4]:**  
- Day48→Day49 holdout R² > 0.90 → submit `submission.csv`. Expect 92-96+ leaderboard.  
- Day48→Day49 holdout R² > 0.80 → still worth submitting (will beat v3 90.66).  
- Day48→Day49 holdout R² < 0.80 → don't submit; the day-invariance hypothesis was wrong.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import pygeohash as pgh

pd.set_option('display.max_columns', 60)
np.random.seed(42)
print('Imports OK')

## Load + Build day-invariant features

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

for df in [train, test]:
    df['hour']    = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']  = df['timestamp'].str.split(':').str[1].astype(int)
    df['ts_mins'] = df['hour']*60 + df['minute']
    coords = df['geohash'].apply(lambda g: pgh.decode_exactly(g)[:2] if pd.notna(g) else (0,0))
    df['gh_lat'] = coords.apply(lambda x: x[0])
    df['gh_lng'] = coords.apply(lambda x: x[1])
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather']  = df['Weather'].fillna('Unknown')
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

rt_map = {v: i for i, v in enumerate(sorted(train['RoadType'].unique()))}
wx_map = {v: i for i, v in enumerate(sorted(train['Weather'].unique()))}

def build_features(df):
    return pd.DataFrame({
        'gh_lat':         df['gh_lat'],
        'gh_lng':         df['gh_lng'],
        'hour':           df['hour'],
        'minute':         df['minute'],
        'ts_mins':        df['ts_mins'],
        'NumberofLanes':  df['NumberofLanes'],
        'Temperature':    df['Temperature'],
        'LargeVehicles':  (df['LargeVehicles'] == 'Allowed').astype(int),
        'Landmarks':      (df['Landmarks']     == 'Yes').astype(int),
        'RoadType_int':   df['RoadType'].map(rt_map).fillna(-1).astype(int),
        'Weather_int':    df['Weather'].map(wx_map).fillna(-1).astype(int),
        # NOTE: `day` deliberately omitted to force day-invariant learning
    })

X_all = build_features(train).values.astype(np.float32)
y_all = train['demand'].values.astype(np.float32)
X_test = build_features(test).values.astype(np.float32)
print(f'Features: {build_features(train).columns.tolist()}')
print(f'X_all: {X_all.shape}  |  X_test: {X_test.shape}')

---
## Sanity Check — Day-48 → Day-49 holdout WITHOUT `day` feature

**This cell decides everything.** Look at `holdout R²`.

In [ ]:
OVERFIT = dict(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=4095,
    max_depth=-1,
    min_child_samples=1,
    min_data_in_leaf=1,
    reg_alpha=0.0,
    reg_lambda=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    n_jobs=-1, random_state=42, verbose=-1,
)

mask_48 = (train['day'] == 48).values
X_d48 = X_all[mask_48];  y_d48 = y_all[mask_48]
X_d49 = X_all[~mask_48]; y_d49 = y_all[~mask_48]

print('Training overfit on day 48 ONLY (no day feature)...')
m = LGBMRegressor(**OVERFIT)
m.fit(X_d48, y_d48)

r2_tr = r2_score(y_d48, m.predict(X_d48))
r2_va = r2_score(y_d49, m.predict(X_d49))
print(f'\nTrain R² = {r2_tr:.4f}   Day48->Day49 holdout R² = {r2_va:.4f}'
      f'   gap = {r2_tr - r2_va:.4f}')

print('\n' + '=' * 60)
if r2_va > 0.90:
    print(f'  >>> EXCELLENT — day-invariance hypothesis confirmed (R²={r2_va:.4f}).')
    print('  >>> Proceeding to fit on full train and generate submission.')
    GO = True
elif r2_va > 0.80:
    print(f'  >>> GOOD — likely to beat v3 90.66 (R²={r2_va:.4f}).')
    print('  >>> Submission still worth it.')
    GO = True
elif r2_va > 0.70:
    print(f'  >>> MARGINAL — might be similar to or worse than v3 (R²={r2_va:.4f}).')
    print('  >>> Submission risky; we will still generate it for you to decide.')
    GO = True
else:
    print(f'  >>> POOR — day-invariance does NOT hold (R²={r2_va:.4f}).')
    print('  >>> Do NOT submit. The 100-scorer is using something else entirely.')
    GO = False
print('=' * 60)

---
## Refit on FULL train + Predict test

Only runs if the sanity check above looked good (GO=True).  
Trains the same overfit config on day 48 + day 49 (full train) and predicts test.

In [ ]:
if not GO:
    print('GO is False. Skipping submission generation.')
else:
    print(f'Refitting overfit model on FULL train ({X_all.shape[0]} rows)...')
    final_model = LGBMRegressor(**OVERFIT)
    final_model.fit(X_all, y_all)
    print('Done.')

    final_preds = final_model.predict(X_test)
    final_preds = np.clip(final_preds, 0.0, 1.0)

    print(f'\nPrediction stats:')
    print(f'  min  = {final_preds.min():.4f}')
    print(f'  max  = {final_preds.max():.4f}')
    print(f'  mean = {final_preds.mean():.4f}  (train mean = {y_all.mean():.4f})')
    print(f'  std  = {final_preds.std():.4f}   (train std  = {y_all.std():.4f})')

    submission = pd.DataFrame({'Index': test['Index'].values, 'demand': final_preds})
    assert submission.shape == (41778, 2), f'Shape {submission.shape}'
    assert submission['demand'].isna().sum() == 0
    assert (submission['Index'].values == test['Index'].values).all()
    submission.to_csv('submission_v11.csv', index=False)

    print(f'\nsubmission_v11.csv saved  |  shape: {submission.shape}')
    print('\nFirst 5 rows:')
    print(submission.head())

---
## Comparison Reference

Feature importances of the final model — to see what it's actually using.

In [ ]:
if GO:
    imp = pd.DataFrame({
        'feature':    build_features(train).columns,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    print('Feature importances (no `day` feature):')
    print(imp.to_string(index=False))

    print(f'\nFinal sanity-check summary:')
    print(f'  v10 Day48->Day49 WITH day feature:    R² = 0.7514 (the bad number)')
    print(f'  v11 Day48->Day49 WITHOUT day feature: R² = {r2_va:.4f}')
    print(f'  Improvement from removing `day`: {r2_va - 0.7514:+.4f}')
    print(f'\n  v3 LightGBM (baseline):  90.66 leaderboard')
    print(f'  v11 expected leaderboard: ~{r2_va*100:.1f} (assuming test ~ day 49 train)')